In [ ]:
import whisper
import spacy
from dotenv import load_dotenv
import os
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

load_dotenv()
# print(os.getenv('GROQ_API_KEY'))

In [ ]:
import langchain
import langchain_core
from langchain.chat_models import init_chat_model

whisper_model = whisper.load_model('medium').to(device)

nlp = spacy.load('en_core_web_sm')

llm= init_chat_model("groq:openai/gpt-oss-20b")

def transcribe_audio(audio_path):
    result=whisper_model.transcribe(audio_path)
    return result['text'], result['language']

In [ ]:
transcript, lang = transcribe_audio('./sample/sample_calls/open_hindi.mp3')

In [ ]:
print(transcript,"\nlan =",lang)

In [ ]:
from langchain_core.output_parsers.string import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

translate_chain = (ChatPromptTemplate([
    ('system','Translate to English. return only translated text.'),
    ('human','translate this {lang} text: {text}')]) | llm | StrOutputParser()
)

english = transcript if lang == 'en' else translate_chain.invoke({'lang': lang,'text':transcript})

In [ ]:
print(str(english))

In [ ]:

doc = nlp(str(english))

persons = []
for e in doc.ents:
    if e.label_ == 'PERSON':
        persons.append(e.text)

urgent_keywords = []
for word in ["urgent", "broken", "cancel", "refund", "angry", "asap"]:
    if word in english.lower():
        urgent_keywords.append(word)


entities = {
    'persons' : persons,
    'urgent_kw' : urgent_keywords
}

In [ ]:
classify_chain = (
    ChatPromptTemplate([
        ('system', ''' 
        
        Classify as CLosed,Open or Urget.

        Closed=resolved
        Open= needs followup
        Urgent = angry/safety/legal.

        return in this format:
        classification: Closed/Open/Urgent
        Reason : one line
        summary : 2-3 lines
        
        '''),
        ('human','Transcirpt: {transcript}\nEntities:{entities}')

        
        
        
    ]) | llm | StrOutputParser()
)

result = classify_chain.invoke({'transcript': english,'entities': entities})
print(result)